In [1]:
import config
# The indices referenced here are with respect to the models in config.MODELS.
# Change these to the models you wish to run the scripts on.
# selected_model_indices = [0, 1, 2]
selected_model_indices = [1]
selected_models = [config.MODELS[i] for i in selected_model_indices]

ModuleNotFoundError: No module named 'config'

In [2]:
from scripts.utils import load_tokenizer, unload_model
from scripts.generate_prompts import SpiderPromptBuilder
builder = SpiderPromptBuilder(config.TABLES_PATH, config.QUESTIONS_PATH)
for model_dict in selected_models:

    tokenizer = load_tokenizer(model_dict)

    prompts_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.PROMPTS_FILENAME}"

    builder.generate_prompts(prompts_path, tokenizer, sample_rate=config.SAMPLE_RATE)
    if model_dict.get("use_in_q3"):
        cot_prompts_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.COT_PROMPTS_FILENAME}"
        builder.generate_prompts(cot_prompts_path, tokenizer, sample_rate=config.SAMPLE_RATE, add_cot=True)

    unload_model(tokenizer)

Generating prompts: 1034it [00:00, 15482.83it/s]


Successfully generated 1034 prompts to data/Qwen/Qwen2.5-Coder-3B-Instruct/formatted_prompts.jsonl


Generating prompts: 1034it [00:00, 20734.10it/s]


Successfully generated 1034 prompts to data/Qwen/Qwen2.5-Coder-7B-Instruct/formatted_prompts.jsonl


Generating prompts: 1034it [00:00, 5236.18it/s]


Successfully generated 1034 prompts to data/Qwen/Qwen3.5-4B/formatted_prompts.jsonl


Generating prompts: 1034it [00:00, 8962.61it/s]

Successfully generated 1034 prompts to data/Qwen/Qwen3.5-4B/cot_formatted_prompts.jsonl


In [ ]:
import config
from scripts.utils import load_model, unload_model
from scripts.get_predictions import LLMPrompter

for model_dict in selected_models:
    print("Loading model "+model_dict['model'])
    model = load_model(model_dict)
    tokenizer = load_tokenizer(model_dict)

    gen_config_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.GEN_CONFIG_FILENAME}"
    prompts_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.PROMPTS_FILENAME}"
    responses_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.RESPONSES_FILENAME}"

    prompter = LLMPrompter(model, tokenizer)
    prompter.load_gen_config(gen_config_path)
    prompter.run_prompt_pipeline(prompts_path, responses_path)

    if model_dict.get("use_in_q3"):
        cot_prompts_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.COT_PROMPTS_FILENAME}"
        cot_responses_path = f"{config.DATA_ROOT}/{model_dict['model']}/{config.COT_RESPONSES_FILENAME}"
        prompter.run_prompt_pipeline(cot_prompts_path, cot_responses_path)

    unload_model(model)
    unload_model(tokenizer)